# Playground Series S6E8「Predicting Smartphone Addiction」: 高スコアnotebook解説

- **コンペ**: [Playground Series - Season 6, Episode 8: Predicting Smartphone Addiction](https://www.kaggle.com/competitions/playground-series-s6e8)
- **元notebook**: [S6E8 honest OOF blend](https://www.kaggle.com/code/szymonkapiski/s6e8-honest-oof-blend) by **szymonkapiski**（Public LB 0.97084、Bronze medal）
- **手法概要**: 74種類のモデル（本人が学習した54個＋他の参加者が公開した20個）の Out-of-Fold（OOF）予測を集めた「ライブラリ」を読み込み、ロジスティック回帰でブレンド（重み付き平均）するだけ、というシンプルな構成。ただし「シンプルなだけに検証が誠実（honest）」であることに全力を注いでいるのがこのnotebookの読みどころ。
- **お断り**: これは学習目的の解説付き写しです。コード自体は改変していませんが、Kaggle上のHTML表示からテキスト取得した際にインデント情報が失われたため、字下げは元のロジックが破綻しないよう書き起こし時に再構成しています。出力（実行結果）はコピー元に含めていません。長大な説明文（著者本人の考察）は要点を残しつつ一部要約しています。

## 評価指標

- **タスク**: 個人の行動・使用状況データから「スマートフォン依存かどうか」（`addicted_label`）を予測する二値分類タスク。
- **指標**: ROC AUC（Receiver Operating Characteristic - Area Under Curve）。予測確率のランキングがどれだけ正解のラベル順序と一致しているかを0〜1で評価する指標で、1に近いほど良い（初心者向け補足: 「陽性/陰性の閾値をどこに引いても、正しく陽性を陰性より高く予測できているか」を測る指標で、クラス不均衡（陽性が少ない/多い）があっても比較的頑健）。
- **なぜこの指標か**: 依存/非依存の判定は閾値の取り方次第で運用側の意図（見逃しを避けたいか、誤検知を避けたいか）が変わるため、特定の閾値に依存しない「順序の正しさ」を測るAUCが適している。また、このnotebookのEDAで触れられている通り学習データは69万行超と大きく、稀な事象を扱うわけではないが、複数モデルの微差を比較する上でAUCは安定した指標になる。
- **この手法がどう指標を最適化しているか**: 単一モデルの精度を上げるのではなく、**相関の低い（=間違え方が異なる）モデルを集めてブレンドする**ことでAUCを底上げしている。さらに「OOF予測を正しいCV分割（fold）で作っているか」を執拗に検証し、リーク（本来学習に使うべきでないデータの情報が混入すること）によるAUCの見せかけの上昇を排除している点が最大の特徴。


### 著者による前置き（要約）

Out-of-fold ROC AUCは0.969687。実際にPublic LBで0.97084を記録した提出は、ここにある74メンバーのうち72個から同様の手順で作られたもので、本notebookが書き出す値とは1e-5以内の差。

このnotebookは何も学習しない。同じCV分割で学習された74モデル分のOOF・test予測を保持する「OOFライブラリ」を読み込み、その上にロジスティック回帰を1つ当てはめるだけ。54個は本人のモデル、20個は同じ分割で動くよう他の参加者の公開モデルを学習し直したもの（19の異なるモデルファミリー）。再学習には数日分のKaggleのGPU/CPU時間を要した。

CPU実行で約17分。Leave-One-Outの診断（`RUN_LOO`）はデフォルトでオフ（74回のブレンド再学習で約1時間かかるため）。

**今回のバージョンの新要素**: 前バージョンは56メンバーでOOF 0.969504・LB 0.97068だった。19個の新メンバー追加で0.96969・0.97084に到達。特に大きいのは、格子状（lattice）勾配ブースティングモデル8個の追加（数値列のペアに対するtarget encodingを0.1刻みの解像度まで広げたもの）と、`lookup`という1つの新モデルファミリー（他のすべてを合わせたより価値がある）。

**この手法の核**: 全モデルが `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` という**全く同じ分割**を使っている。これにより、異なる人が書いた異なるnotebookのOOF予測でも、正直にスタッキング（積み上げ）できる。


### 「正直なOOF」であることの重要性（要約）

著者が見つけたS6E8のOOF公開notebookの多くは実は使い物にならなかった：あるシリーズは7-fold、別のものは3つの乱数シードで平均していて分割が3通り混ざっている、`lookup`アーキテクチャは10-foldを使用——これらはどれも「一見健全なOOF配列（長さも範囲もそれらしい）」に見えるが、実際には**リーク**を含み、それに気づかず使うとブレンドのスコアが不当に良く見えてしまう。

統計的な検定（KS検定など）でもこの種のリークはほとんど検出できないことを著者は実際に確認している。**唯一の対策は、コードを読んで `n_splits` と `random_state` を確認すること**。

### 相関と多様性についての教訓（要約）

「相関が高い（似た間違え方をする）モデルは、スコアが高くてもブレンドへの寄与はゼロに近い」——実際に`lat_xgb`はOOF 0.96749というそこそこのスコアだが、既存モデルと相関0.998のためブレンドへの寄与はほぼゼロ。逆に`pubmk_nn`は単体スコア0.94085と低いが、相関0.906とパックの中では相対的に独立しており、部分集合を選ぶなら「強いモデル」より「意見が割れるモデル」を選ぶべき、という考察。

### 精度についての教訓（要約）

予測配列をfloat32にダウンキャストすると、単体モデルのAUC（順位のみに依存）は変わらないが、ブレンドの結果には影響する。メンバー同士の相関が0.99以上と高く、メタモデルは0と1付近のfloat32の分解能より小さい差を扱っているため、実際にダウンキャストでテスト行の順位の28%が入れ替わり、LBスコアが0.00001悪化した、という実測に基づく注意点。


In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


# glob rather than hardcode: Kaggleは環境によって入力データのマウント方法が微妙に違うことがあるため、
# 決め打ちせずglobで探索する。
def find_library():
    for p in glob.glob("/kaggle/input/**/oof_lat_lgbm.npy", recursive=True):
        return os.path.dirname(p)
    raise FileNotFoundError("OOF library not attached")


def find_comp():
    for h in glob.glob("/kaggle/input/**/test.csv", recursive=True):
        if os.path.exists(os.path.join(os.path.dirname(h), "train.csv")):
            return os.path.dirname(h)
    raise FileNotFoundError("competition data not attached")


LIB, COMP = find_library(), find_comp()
print("library:", LIB)

te_ids = pd.read_csv(f"{COMP}/test.csv", usecols=["id"])
y = pd.read_csv(f"{COMP}/train.csv", usecols=["addicted_label"])["addicted_label"].values

names = sorted(os.path.basename(p)[4:-4] for p in glob.glob(f"{LIB}/oof_*.npy"))
print(f"{len(names)} members")

O, T = [], []
for n in names:
    o = np.load(f"{LIB}/oof_{n}.npy").astype("float64")
    t = np.load(f"{LIB}/test_{n}.npy").astype("float64")
    # ロード時点での形状チェックは軽い処理だが、メンバーの配列がずれていた場合に
    # ブレンド全体を静かに壊してしまう前にここで検知できる。
    assert o.shape == (len(y),), (n, o.shape)
    assert t.shape == (len(te_ids),), (n, t.shape)
    O.append(o)
    T.append(t)
O, T = np.column_stack(O), np.column_stack(T)
print("stacked:", O.shape, T.shape)


**何をしているか**: 74個のモデルそれぞれのOOF予測（`oof_*.npy`）とtest予測（`test_*.npy`）をすべて読み込み、1つの行列（サンプル数 × モデル数）にまとめる。
**なぜそうするのか**: `assert`で形状チェックを入れているのは地味だが重要な工夫。あるモデルの配列だけ長さがずれていた場合、そのままブレンドに混ぜると気づかないうちに結果全体を壊してしまう。読み込み時点で検知するコストは小さいので、早い段階でチェックする設計になっている（初心者向け補足: `assert`文は「条件を満たさなければ即座にエラーを出す」防御的プログラミングの基本テクニック）。

In [ ]:
auc = {n: roc_auc_score(y, O[:, i]) for i, n in enumerate(names)}
tab = pd.DataFrame({"model": names, "oof_auc": [auc[n] for n in names]})
tab = tab.sort_values("oof_auc", ascending=False).reset_index(drop=True)
print(tab.head(12).to_string(index=False))
print(f"\nbest single: {tab.iloc[0]['model']} {tab.iloc[0]['oof_auc']:.5f}")


**何をしているか**: 74モデルそれぞれの単体OOF AUCを算出し、ランキング表示する。
**なぜそうするのか**: 全モデルが同じfold分割で学習されているため、この単体スコアの比較は「同じ土俵での比較」として意味を持つ（分割が違うモデル同士のスコアを比べても正確な優劣は分からない）。最良単体モデルは0.96881だが、後のセルでブレンドすると0.9697近くまで伸びることが示される。

**プレーンブレンド（要約）**: 74モデルすべてのlogit（対数オッズ）にロジスティック回帰を1本当てはめるだけの、最初のシンプルなブレンド。確率のままではなくlogit空間で平均するのは、確率を対数オッズに変換することでモデルの組み合わせが「対数オッズの加重平均」という自然な形になるため。

**Honest（正直な）CVという設計**: メタモデル（ブレンド用のロジスティック回帰）は各foldの中で学習し直し、そのfoldでは学習に使っていない行だけを予測する。全データで1回だけ学習してそのまま同じデータを採点すると、in-sample（学習に使ったデータそのもの）の甘い数値になってしまうため、この「fold内で学習→held-outだけ予測」を徹底している。

In [ ]:
# logit空間で作業する。これはメタモデルが対数オッズを平均するのに自然なスケールだから。
EPS = 1e-6
OL = logit(np.clip(O, EPS, 1 - EPS))
TL = logit(np.clip(T, EPS, 1 - EPS))

# Honest blend CV: メタモデルは各fold内で学習し直し、学習に使っていない行だけを予測する。
# 全OOFで1回学習してそのまま同じ行を採点すると、in-sampleの甘い数値になってしまう。
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
blend_oof = np.zeros(len(y))
for itr, iva in skf.split(OL, y):
    lr = LogisticRegression(max_iter=3000, C=1.0).fit(OL[itr], y[itr])
    blend_oof[iva] = lr.predict_proba(OL[iva])[:, 1]

blend_cv = roc_auc_score(y, blend_oof)
best_single = max(auc.values())
print(f"blend OOF AUC {blend_cv:.5f}")
print(f"best single {best_single:.5f}")
print(f"gain {blend_cv - best_single:+.5f}")


**ウェイト（重み）についての著者の考察（要約）**: いくつかのモデルの重みは負になる。バグではない——メンバー同士の相関が0.99以上と非常に高いため、メタモデルは一部のモデルを「独立した意見」ではなく「補正項」として使っている。重みを非負に制約するとスコアが悪化することを著者は確認済み。

In [ ]:
final = LogisticRegression(max_iter=3000, C=1.0).fit(OL, y)
w = pd.DataFrame({"model": names, "weight": final.coef_[0]})
w["abs"] = w["weight"].abs()
print(w.sort_values("abs", ascending=False).head(12)[["model", "weight"]].to_string(index=False))


**Leave-One-Out診断（要約）**: 各メンバーを1つ抜いてブレンドを再学習し、スコアがどれだけ落ちるかを見ることで「そのモデルを学習する計算コストに見合う価値があるか」を判定する。74回のブレンド再学習が必要で約1時間かかるため、デフォルトでは`RUN_LOO=False`にして過去の実行結果を文字列として貼り付けてある。結果：`lookup`を抜くと-0.000106と突出して大きく落ちる。他の73モデルは「個々の強さ」ではなく「多様性（breadth）」が効いている、という結論。

In [ ]:
def blend_cv_of(cols):
    M = OL[:, cols]
    b = np.zeros(len(y))
    for itr, iva in skf.split(M, y):
        b[iva] = LogisticRegression(max_iter=3000).fit(M[itr], y[itr]).predict_proba(M[iva])[:, 1]
    return roc_auc_score(y, b)


# 74回のブレンド再学習、約1時間。実行はデフォルトでオフ。
RUN_LOO = False

if RUN_LOO:
    full_idx = list(range(len(names)))
    rows = [(n, blend_cv_of([j for j in full_idx if j != i]) - blend_cv)
            for i, n in enumerate(names)]
    loo = pd.DataFrame(rows, columns=["model", "delta_if_dropped"]).sort_values("delta_if_dropped")
    print(loo.head(12).to_string(index=False))
    print(f"\nmembers costing less than 0.00002 to drop: {(loo.delta_if_dropped > -0.00002).sum()} of {len(loo)}")
else:
    print("RUN_LOO = False. Stored result, full 74-member run of this cell:")
    print("""
         model  delta_if_dropped
        lookup         -0.000106
    pub_tabnet         -0.000016
      pub_tabm         -0.000016
     pub_ryota         -0.000015
     latr1_xgb         -0.000009
   tabm_deeper         -0.000009
            et         -0.000007
           cat         -0.000007
      pubfe_xgb        -0.000005
       pub_evg         -0.000005
   lat_lgbm_s5         -0.000005
     pubmk_nn          -0.000004

    members costing less than 0.00002 to drop: 73 of 74
    """)

print("\nOne member is load-bearing and it is lookup, at 6.6 times the next one.")
print("For the other 73, breadth is what helps, not any individual model.")


### `lookup`モデルについて（要約）

ブレンドが飽和していた（スコア0.966以上のモデルは互いに相関0.986〜0.999）ため、「別の特徴量の見方」を作ろうと4種類のview（rank変換、残差化、target encoding無し、生成器の制約から導く区間）を試したが、8モデル合計でもゲインは+0.000007しかなかった。

対照的に`lookup`はたった1モデルで、8個のview合計の15倍のゲイン（+0.000109）を生んだ。理由は「データが合成データであり、数値列は実は数千個の値が繰り返される格子（lattice）である」という構造に対し、target encodingが格子セルを1つのスカラーに潰すのに対し、`lookup`は観測された値それぞれに128次元の埋め込みを学習で割り当てるという、**同じ構造を別の仕組みで読む**アプローチだったこと。アーキテクチャ自体は別の参加者（tamerlanomralinov氏）のものだが、その人の公開OOFは10-foldで作られておりこの5-fold分割とは互換性がないため、著者はアーキテクチャだけを流用してこの5-fold分割で**再学習**している（約40分・T4 GPU1枚）。


In [ ]:
# lookupが「決定的に良いモデルというより、決定的に相関が低いモデル」だという主張の検証。
if "lookup" in names:
    i = names.index("lookup")
    c = np.array([np.corrcoef(OL[:, i], OL[:, j])[0, 1] for j in range(len(names)) if j != i])
    other = [n for j, n in enumerate(names) if j != i]
    order = np.argsort(-c)
    print(f"lookup OOF AUC {auc['lookup']:.5f}")
    print(f"MAX correlation {c.max():.4f} ({other[order[0]]})")
    print("\nnearest five:")
    for k in order[:5]:
        print(f"  {other[k]:20s} corr {c[k]:.4f} oof {auc[other[k]]:.5f}")

    # 比較用: lookup以外の強いモデル群は、互いにどれくらい相関しているか？
    strong = [j for j, n in enumerate(names) if auc[n] >= 0.966 and n != "lookup"]
    mx = []
    for a in strong:
        mx.append(max(np.corrcoef(OL[:, a], OL[:, b])[0, 1] for b in strong if b != a))
    print(f"\nmembers at OOF >= 0.966, excluding lookup: {len(strong)}")
    print(f"  their max-correlation, median {np.median(mx):.4f}, min {min(mx):.4f}")
    print(f"  lookup's max-correlation {c.max():.4f}")
else:
    print("lookup not in this library version")


### 欠損状況で重みを変える「missingness regime」設計（要約）

すべてのモデルは欠損の少ない行で著しく精度が良い。ならば「行の欠損具合」に応じてメタモデルがメンバーの重み付けを変えられるようにしよう、というのがriponce氏考案の設計。目的変数の陽性率は欠損グループ間でほぼ変わらない（MCAR＝完全にランダムな欠損）にもかかわらず、「モデルの当てにできる度合い」は欠損具合で大きく変わる（完全な行でAUC 0.977、4個以上欠損の行でAUC 0.930）という点がポイント。

In [ ]:
# 欠損状況に応じた重み付け設計（riponce氏考案）。通常のメンバーlogitに加え、
# 「完全な行か」「4つ以上欠損した行か」「メンバー間の意見の割れ具合」との交互作用項を加える。
feat_cols = [c for c in pd.read_csv(f"{COMP}/test.csv", nrows=1).columns if c != "id"]
miss_tr = pd.read_csv(f"{COMP}/train.csv", usecols=feat_cols).isna().sum(axis=1).values
miss_te = pd.read_csv(f"{COMP}/test.csv", usecols=feat_cols).isna().sum(axis=1).values

# 目的変数の陽性率はレジーム間でほぼ動かない（MCAR）が、メンバーの「信頼度」は大きく動く。
# 重要なのは後者だけ。
for lab, m in [("complete", miss_tr == 0), ("1-3 missing", (miss_tr >= 1) & (miss_tr <= 3)),
               ("4+ missing", miss_tr >= 4)]:
    print(f"{lab:12s} n={m.sum():>7,} target rate {y[m].mean():.5f} "
          f"blend AUC {roc_auc_score(y[m], blend_oof[m]):.5f}")


def regime_design(lg, missing, dmean=None, dstd=None):
    complete = (missing == 0).astype("float64")[:, None]
    severe = (missing >= 4).astype("float64")[:, None]
    d = lg.std(axis=1, keepdims=True)
    # test側の設計はTRAIN側の統計量で正規化する（自分自身の統計量では正規化しない）。
    # test予測は5-foldモデルの平均なのでばらつきが系統的に小さくなるため。
    if dmean is None:
        dmean, dstd = float(d.mean()), float(d.std())
    dn = (d - dmean) / (dstd + 1e-6)
    agg = np.column_stack([lg.mean(1), lg.std(1), lg.max(1) - lg.min(1), complete[:, 0], severe[:, 0]])
    return np.column_stack([lg, lg * complete, lg * severe, lg * dn, agg]), dmean, dstd


MR, dmean, dstd = regime_design(OL, miss_tr)
MRT, _, _ = regime_design(TL, miss_te, dmean, dstd)
print(f"\nglobal design {OL.shape} -> regime design {MR.shape}")


**正則化の強さ選び（要約）**: フルグリッドサーチは重いので、その形状が既に分かっている前提でスキップしている。過去にフルグリッドを回した結果、`C`（正則化の弱さ）を大きくするほどわずかに改善が続き、riponce氏が採用した`C=0.1`はグリッドの端であり最適点ではなかったことが判明。加えて上位4点の差は百万分の1程度とほぼフラット。さらに、ライブラリのメンバー数が変わると最適な`C`も動く（56メンバーで1.0、66メンバーで0.1、74メンバーで0.03）ため、値を固定でハードコードするのは危険——という理由で、2点だけ確認して片方を選ぶ設計にしている。

In [ ]:
from sklearn.preprocessing import StandardScaler

# 全グリッドではなく、かつ固定値でもない2点。理由: Cカーブの頂上は百万分の1程度でほぼフラットだが、
# その頂上（argmax）はメンバー構成が変わるたびに動く(56メンバーで1.0, 66メンバーで0.1, 74メンバーで0.03)。
# 2点あれば、その「頂上に乗っているか」を確認できる。
C_GRID = [0.03, 0.1]
# tol=1e-5 (1e-6ではなく): この設計で計測したところOOFは小数点6桁まで同一で、実行時間は880s→500sに短縮。
TOL = 1e-5


def honest_oof(X, C, tol=TOL):
    pred = np.zeros(len(y))
    nit = []
    for itr, iva in skf.split(X, y):
        sc = StandardScaler().fit(X[itr])
        m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=tol)
        m.fit(sc.transform(X[itr]), y[itr])
        nit.append(int(np.max(m.n_iter_)))
        pred[iva] = m.predict_proba(sc.transform(X[iva]))[:, 1]
    return pred, roc_auc_score(y, pred), max(nit)


# グローバルベースラインもレジームモデルと同じスケーラー・ソルバー設定で再計算し、
# 公平な比較（like-for-like）にする。
global_oof, global_auc, _ = honest_oof(OL, 1.0)
print(f"global stack {global_auc:.6f}")

best = (-1, None, None)
for C in C_GRID:
    p, a, it = honest_oof(MR, C)
    print(f"regime C={C:<6g} {a:.6f} (lbfgs iters <= {it})")
    if a > best[0]:
        best = (a, C, p)

regime_auc, best_C, regime_oof = best
gain = regime_auc - global_auc
print(f"\nglobal {global_auc:.6f} | regime {regime_auc:.6f} at C={best_C} | gain {gain:+.6f}")

# riponce氏のフォールバックルール: 小さな正直なマージンを超えられない、より柔軟なモデルは
# 「改善」とはみなさない。
use_regime = gain >= 0.00002
print("selected:", "regime-aware" if use_regime else "global (fallback)")


**どこで効いているか（要約）**: 56メンバー版のライブラリでは、regime設計は「綺麗な行」でゲインがあり、本来助けるはずの「4つ以上欠損」グループでは逆に悪化していた——著者はこれを「正しい理由ではない理由で効いているケース」として一度は結論づけた。しかし`lookup`が加わった現在のライブラリでは、逆に「4つ以上欠損」グループが最もゲインしている。理由として最も有力なのは、`lookup`が学習時に欠損マスキング拡張（値をランダムに隠す）を行っており、これが重度に欠損した行での重み付け直しにとって本当に価値のある情報をメタモデルに与えているため、という考察。著者は「古い解釈が間違っていた」と黙って消すのではなく、「当時のデータからは正直な結論だった」として明記して残している。

In [ ]:
# どこで実際に効いているか。lookupがライブラリに加わって答えが変わった。
for lab, m in [("complete", miss_tr == 0), ("1-3 missing", (miss_tr >= 1) & (miss_tr <= 3)),
               ("4+ missing", miss_tr >= 4)]:
    g = roc_auc_score(y[m], global_oof[m])
    r = roc_auc_score(y[m], regime_oof[m])
    print(f"{lab:12s} n={m.sum():>7,} global {g:.6f} regime {r:.6f} delta {r - g:+.6f}")


In [ ]:
# 両方のメタモデルを全学習データで学習し直し、選ばれた方を提出ファイルとして書き出す。
sc_g = StandardScaler().fit(OL)
final_global = LogisticRegression(C=1.0, max_iter=5000, solver="lbfgs", tol=TOL)
final_global.fit(sc_g.transform(OL), y)
pred_global = final_global.predict_proba(sc_g.transform(TL))[:, 1]

sc_r = StandardScaler().fit(MR)
final_regime = LogisticRegression(C=best_C, max_iter=5000, solver="lbfgs", tol=TOL)
final_regime.fit(sc_r.transform(MR), y)
pred_regime = final_regime.predict_proba(sc_r.transform(MRT))[:, 1]

pred = pred_regime if use_regime else pred_global
sub = pd.DataFrame({"id": te_ids["id"], "addicted_label": pred})
assert len(sub) == len(te_ids)
assert sub["addicted_label"].between(0, 1).all()
assert sub["addicted_label"].nunique() > 1000
sub.to_csv("submission.csv", index=False)
print(sub.shape, "mean pred", round(float(pred.mean()), 5))
sub.head()


**何をしているか**: プレーンなグローバルブレンドと、欠損状況を考慮したregimeブレンドの両方を全データで学習し、`gain >= 0.00002`という小さな閾値をクリアした方（今回はregime版）を最終予測として`submission.csv`に書き出す。
**なぜそうするのか**: 「複雑にしたモデルが必ずしも良いとは限らない」という前提のもと、正直なCVで確認できた改善だけを採用するという一貫した姿勢がこのnotebook全体を貫いている。最後にも`between(0, 1)`や`nunique() > 1000`のような提出直前のバリデーション（初心者向け補足: 想定外の値や定数だらけの提出物を防ぐ最後の砦）を入れている。

### 統計的な考察: リーダーボードで判別できる限界（要約）

著者はブートストラップ法でスコアのノイズの大きさそのものを測定している。実際のtestサイズ（296,302行、陽性率0.709）にリサンプリングした結果、**単体のAUCスコアの標準偏差は約0.000272**、一方で**2つの相関したサブミッション同士の差の標準偏差は約0.000072**（単体スコアの誤差の約4分の1）。この違いは、両方のサブミッションが同じtest行で採点されるため、「どの行がtestに入ったか」由来のブレがこの差の中でキャンセルするため（数式では `sqrt(2(1-rho))` という係数で説明でき、メンバー同士の相関0.99のもとでは差はかなり小さくなる）。

結論として、**0.00014より小さい変化はPublicリーダーボードでは判別できない**。本notebookで積み上げている改善の多く（regimeで+0.000029、格子ペアで+0.000059など）はこの閾値を下回っており、69万行超のOOFセットがあって初めて統計的に検出できるレベルの改善である。だからこそ著者は「選定はCVで行い、リーダーボードは健全性チェックにすぎない」という姿勢を取っており、CVとLBのオフセットは複数回の提出を通じて安定して+0.0012前後だったという。


## まとめ

このnotebookの本当の学びどころは「74モデルのブレンド」という規模そのものではなく、**(1) 全メンバーが同一のCV分割を使っているかを執拗に確認する誠実さ、(2) 統計的に意味のある改善とノイズの範囲内の変化を区別する姿勢、(3) 古い結論が新しいデータで覆ったときにそれを隠さず記録する透明性**にあります。派手な単一モデルの工夫よりも、「検証の設計そのものを疑い、検証する」という地道な作業が最終的なスコアの信頼性を支えている好例です。